In [ ]:
pip install agentpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 524.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.8/778.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00


# Ambiente  

*   `GridEnvironment` define una rejilla con una celda objetivo que tiene una recompensa de 100. Las demás celdas tienen una recompensa de -1.
*   `move_agent` mueve al agente en la rejilla según la acción elegida.

In [ ]:
import agentpy as ap
import numpy as np
import random

class GridEnvironment(ap.Grid):
    def setup(self):
        self.grid_size = self.p.grid_size
        self.objetivo = self.p.objetivo
        self.grid = np.zeros((self.grid_size, self.grid_size))
        self.grid[self.objetivo] = 100  # Recompensa en la celda objetivo

    def get_reward(self, state):
        if state == self.objetivo:
            return 100
        else:
            return -1

    def move_agent(self, state, action):
        x, y = state
        if action == 'arriba' and x > 0:
            x -= 1
        elif action == 'abajo' and x < self.grid_size - 1:
            x += 1
        elif action == 'izquierda' and y > 0:
            y -= 1
        elif action == 'derecha' and y < self.grid_size - 1:
            y += 1
        return (x, y)

In [ ]:
class QLearningAgent(ap.Agent):
    def setup(self):
        self.actions = ['arriba', 'abajo', 'izquierda', 'derecha']
        self.Q = {}
        for x in range(self.p.grid_size):
            for y in range(self.p.grid_size):
                self.Q[(x, y)] = {action: 0 for action in self.actions}
        self.epsilon = self.p.epsilon
        self.alpha = self.p.alpha
        self.gamma = self.p.gamma

    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions)
        else:
            return max(self.Q[state], key=self.Q[state].get)

    def update_Q(self, state, action, reward, new_state):
        max_Q_new_state = max(self.Q[new_state].values())
        self.Q[state][action] = self.Q[state][action] + self.alpha * (
            reward + self.gamma * max_Q_new_state - self.Q[state][action])

#Modelo:


In [ ]:
class QLearningModel(ap.Model):
    def setup(self):
        self.env = GridEnvironment(self, shape=(self.p.grid_size, self.p.grid_size))
        self.agent = QLearningAgent(self)

    def step(self):
        state = (0, 0)
        while state != self.p.objetivo:
            action = self.agent.choose_action(state)
            new_state = self.env.move_agent(state, action)
            reward = self.env.get_reward(new_state)
            self.agent.update_Q(state, action, reward, new_state)
            state = new_state

    def end(self):
        self.report('Q-Table', self.agent.Q)


In [ ]:
parameters = {
    'grid_size': 5,
    'objetivo': (4, 4),
    'epsilon': 0.1,
    'alpha': 0.1,
    'gamma': 0.9,
    'steps': 100
}

model = QLearningModel(parameters)
results = model.run()


# Visualization
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
import IPython
import random
import json
import numpy as np

def animation_plot(model, ax):
    # Grafica el máximo valor de cada acción de acuerdo a la celda
    U, D, L, R = [np.array([model.agent.Q[i,j][act] for i in range(5) for j in range(5)]).reshape((5,5)) for act in ['arriba', 'abajo', 'izquierda', 'derecha']]
    grid = np.maximum(U, D)
    grid = np.maximum(grid, L)
    grid = np.maximum(grid, R)
    ap.gridplot(grid, cmap='Greys', ax=ax)
    ax.set_title(f"Agent Q-Learning")
                 #f"Dirty cells: {model.get_dirty_cells()}")

fig, ax = plt.subplots()
model = QLearningModel(parameters)
animation = ap.animate(model, fig, ax, animation_plot)
IPython.display.HTML(animation.to_jshtml())

Completed: 100 steps
Run time: 0:00:00.053492
Simulation finished


In [ ]:
np.array([model.agent.Q[i,j]['abajo'] for i in range(5) for j in range(5)]).reshape((5,5))

array([[-1.38602787, 36.77306139, 12.00720968, -0.58963843, -0.16974367],
       [-1.48741358, -0.21169782, 58.4721114 , 21.43482626,  5.73053042],
       [-1.0438032 , -0.7094876 , -0.4794561 ,  1.07195527, 88.89855349],
       [-0.71801742, -0.3389328 ,  3.33474135, 30.69100683, 99.98566589],
       [-0.4900995 , -0.29701   , -0.199     , -0.1       ,  0.        ]])